In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, ConfusionMatrixDisplay,
    RocCurveDisplay, confusion_matrix
)
from preprocessing import load_data, split_data, make_preprocessor, FEATURES, RANDOM_STATE

FIG_DIR = Path("results/logistic_regression")
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
X, y = load_data()
X_train, X_test, y_train, y_test = split_data(X, y)
print("Total records:", len(X))
print("Training records:", len(X_train), "Test records:", len(X_test))
print("Training diabetes rate:", round(y_train.mean(), 3))
print("Test diabetes rate:", round(y_test.mean(), 3))
print("Missing predictor values (after zero replacement):")
display(X.isna().sum())

Total records: 768
Training records: 614 Test records: 154
Training diabetes rate: 0.349
Test diabetes rate: 0.351
Missing predictor values (after zero replacement):


Preg          0
Glucose       5
BP           35
Skin        227
Insulin     374
BMI          11
Pedigree      0
Age           0
dtype: int64

In [ ]:
grid = GridSearchCV(
    make_lr_pipeline(),
    param_grid={"lr__C": [0.01, 0.1, 1, 10, 100],
                "lr__class_weight": [None, "balanced"]},
    scoring=scoring, refit="roc_auc", cv=cv, n_jobs=-1,
    return_train_score=True
)
grid.fit(X_train, y_train)
print("Best hyperparameters:", grid.best_params_)
print("Best CV ROC-AUC:", round(grid.best_score_, 3))

best_idx = grid.best_index_
comparison = pd.DataFrame({
    "Baseline CV": baseline_summary,
    "Tuned CV": pd.Series({m: grid.cv_results_[f"mean_test_{m}"][best_idx]
                            for m in scoring})
}).round(3)
display(comparison)